# 03 · Prices, market cap & trading calendar (Data layer)

Builds the **price store** everyone else uses:

| Output (in `Data/prices/`) | What it is |
|---|---|
| `prices.parquet` | one row per ticker per trading day: open/high/low/close, **adj_close**, volume, daily return, **raw (unadjusted) close**, shares outstanding, **market cap**, size bucket |
| `benchmarks.parquet` | same, for SPY (main benchmark) + IWM (small caps) + MDY (mid caps) |
| `dividends_yf.parquet` | dividend ex-dates and amounts from Yahoo, for cross-checking the LLM extraction |
| `splits_yf.parquet` | stock splits |
| `shares_outstanding.parquet` | point-in-time share counts from SEC filings (dated by when they were **filed**) |
| `trading_calendar.csv` | NYSE trading days (from SPY) |
| `price_coverage.csv` | per-ticker coverage + data-quality flags |
| `../universe_quality.csv` | per company: **keep / keep_from / drop** + `usable_from` date (uses notebook 01's filing index) |

Reads the ticker list from `universe.csv` and the filing index (both made by notebook 01).

**Setup:** set `USER_AGENT` below. The data folder comes from `$GENAI_DATA_DIR`, or in Colab from the mounted Drive.
Run top to bottom; safe to re-run (about 15 minutes). The helpers in section 10 are the same as `src/store/prices.py`.

## 1. Config

In [ ]:
USER_AGENT = "Your Name your.email@example.com"   # required by SEC (for shares outstanding): your name + email

# Start before 2016 so backtests have a look-back window (e.g. 60-day volume, estimation windows)
PRICE_START = "2015-01-01"
PRICE_END   = None            # None = up to today

BENCHMARKS = ["SPY", "IWM", "MDY"]   # S&P 500 (main), Russell 2000 (small caps), S&P 400 (mid caps)

# Size buckets: "tercile" = split the universe into thirds by market cap on each date (point-in-time, recommended)
#               "fixed"   = fixed USD cut-offs below
SIZE_METHOD = "tercile"
MCAP_BUCKETS = {"small": (0, 2e9), "mid": (2e9, 10e9), "large": (10e9, float("inf"))}   # only for "fixed"

# Data-quality rules -> is_bad_price (Backtest should skip these days)
EXTREME_RETURN = 0.5          # reported in checks
BAD_RETURN = 1.0              # |daily return| above this is treated as a data error
NEW_LISTING_DAYS = 10         # first N trading days of a new listing / spin-off are unreliable
BAD_RANGES = {                # known broken histories: ticker -> (from, to) inclusive, None = open-ended
    "CHRD": (None, "2020-11-19"),   # Oasis Petroleum bankruptcy; old shares cancelled, new equity listed 2020-11-20
}

# Universe quality (uses Data/index/filing_index.parquet from notebook 01)
MIN_FILINGS = 20              # fewer 8-Ks than this over the period -> drop (e.g. banks that file with the FDIC)
LATE_FILINGS_GAP_DAYS = 365   # filings start > 1 year after prices -> company probably changed SEC ID; usable only from then

# Data folder (the shared Drive folder "Gen AI/Data/"):
#   - uses $GENAI_DATA_DIR if set (same variable as the repo's code), e.g. your Google Drive for desktop path
#   - otherwise, in Colab, mounts Drive and uses the path below (needs a "Gen AI" shortcut in My Drive)
COLAB_DATA_DIR = "/content/drive/MyDrive/Gen AI/Data"

BATCH_SIZE = 40               # tickers per Yahoo request
MAX_REQUESTS_PER_SEC = 8      # SEC limit is 10

## 2. Setup

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "pandas", "pyarrow", "requests"], check=False)

In [ ]:
import time, pathlib
import numpy as np
import pandas as pd
import requests
import yfinance as yf

import os
if "your.email@example.com" in USER_AGENT:
    raise ValueError("Set USER_AGENT in the config cell to your name and email (SEC requires it).")

MOUNTED = False
if os.environ.get("GENAI_DATA_DIR"):
    BASE = pathlib.Path(os.environ["GENAI_DATA_DIR"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MOUNTED = True
    BASE = pathlib.Path(COLAB_DATA_DIR)
else:
    raise RuntimeError("Set GENAI_DATA_DIR to your local copy of the shared Drive folder 'Gen AI/Data/' (see README).")

OUT = BASE / "prices"
OUT.mkdir(parents=True, exist_ok=True)

universe = pd.read_csv(BASE / "universe.csv", dtype={"cik": str})
universe["cik"] = universe["cik"].str.zfill(10)
TICKERS = universe["ticker"].tolist()
print(f"{len(TICKERS)} tickers from universe.csv | saving to {OUT.resolve()}")

## 3. Download daily prices from Yahoo

`auto_adjust=False` keeps both prices:
- **`close`** — adjusted for splits only (what the chart shows)
- **`adj_close`** — adjusted for splits **and** dividends → use this for **returns**

Downloaded in batches; any ticker that fails is retried on its own.

In [ ]:
FIELDS = {"Open": "open", "High": "high", "Low": "low", "Close": "close", "Adj Close": "adj_close",
          "Volume": "volume", "Dividends": "dividend", "Stock Splits": "split_ratio"}

def _to_long(data, tickers):
    """yfinance wide frame -> long rows (date, ticker, fields...)."""
    if not isinstance(data.columns, pd.MultiIndex):           # single ticker, flat columns
        data = pd.concat({tickers[0]: data}, axis=1)
    lvl0 = set(data.columns.get_level_values(0))
    if not set(tickers) & lvl0:                               # (field, ticker) order -> swap
        data = data.swaplevel(0, 1, axis=1)
    frames = []
    for t in tickers:
        if t not in set(data.columns.get_level_values(0)):
            continue
        sub = data[t].rename(columns=FIELDS)
        sub = sub[[c for c in FIELDS.values() if c in sub.columns]].dropna(subset=["adj_close"])
        if len(sub):
            sub = sub.reset_index().rename(columns={"Date": "date", "index": "date"})
            sub.insert(1, "ticker", t)
            frames.append(sub)
    return frames

def download_prices(tickers):
    frames, failed = [], []
    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i + BATCH_SIZE]
        try:
            data = yf.download(batch, start=PRICE_START, end=PRICE_END, auto_adjust=False, actions=True,
                               group_by="ticker", threads=True, progress=False)
            got = _to_long(data, batch)
        except Exception as e:
            print("batch error:", e)
            got = []
        frames += got
        got_t = {f["ticker"].iat[0] for f in got}
        failed += [t for t in batch if t not in got_t]
        print(f"{min(i + BATCH_SIZE, len(tickers))}/{len(tickers)} tickers")
        time.sleep(1)
    for t in list(failed):                                    # retry one by one
        try:
            data = yf.download(t, start=PRICE_START, end=PRICE_END, auto_adjust=False, actions=True, progress=False)
            got = _to_long(data, [t])
            if got:
                frames += got
                failed.remove(t)
        except Exception:
            pass
        time.sleep(1)
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()
    return df.sort_values(["ticker", "date"]).reset_index(drop=True), failed

px, failed = download_prices(TICKERS)
bm, bm_failed = download_prices(BENCHMARKS)
print(f"\nStocks: {px.ticker.nunique()} tickers, {len(px):,} rows | failed: {failed}")
print(f"Benchmarks: {sorted(bm.ticker.unique())} | failed: {bm_failed}")

## 4. Trading calendar + returns

The calendar comes from SPY's trading days. Notebook 02 uses it to compute `first_tradeable_date`.

In [ ]:
calendar = pd.DatetimeIndex(sorted(bm.loc[bm.ticker == "SPY", "date"].unique()))
pd.DataFrame({"date": calendar.date}).to_csv(OUT / "trading_calendar.csv", index=False)
print(f"Trading calendar: {calendar[0].date()} -> {calendar[-1].date()} ({len(calendar):,} days)")

for d in (px, bm):
    d["ret"] = d.groupby("ticker")["adj_close"].pct_change()

## 5. Undo split adjustment → raw close

Market cap = **shares outstanding (as reported at the time) × price at the time**.
Yahoo's `close` is split-adjusted, so before a split it doesn't match the share count reported then.
This multiplies back the splits that happened *after* each date.

In [ ]:
def add_raw_close(d):
    d = d.sort_values(["ticker", "date"]).copy()
    ratio = d["split_ratio"].fillna(0).replace(0, 1.0)
    # factor(date) = product of split ratios strictly AFTER that date
    rev_cum = ratio[::-1].groupby(d["ticker"][::-1]).cumprod()[::-1]
    d["split_factor"] = (rev_cum / ratio).values
    d["raw_close"] = d["close"] * d["split_factor"]
    return d

px = add_raw_close(px)
splits = px.loc[px["split_ratio"].fillna(0) > 0, ["date", "ticker", "split_ratio"]]
print(f"{len(splits)} splits across {splits.ticker.nunique()} tickers")

## 6. Shares outstanding from SEC (point-in-time)

From SEC's free XBRL "company facts" API: the cover-page share count on each 10-K/10-Q.
Each value is dated by **when it was filed** (not the period end), so we never use a number before the market could know it.

In [ ]:
import threading
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})
_lock, _next = threading.Lock(), [0.0]

def sec_get_json(url):
    for attempt in range(3):
        with _lock:
            now = time.time(); slot = max(now, _next[0]); _next[0] = slot + 1 / MAX_REQUESTS_PER_SEC
        if slot > now:
            time.sleep(slot - now)
        r = SESSION.get(url, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 404:
            return None
        if r.status_code == 403:
            raise RuntimeError("403 from SEC - check USER_AGENT (or run locally if Colab is blocked)")
        time.sleep(2 ** attempt)
    return None

def shares_history(cik):
    facts = sec_get_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json")
    if not facts:
        return pd.DataFrame()
    f = facts.get("facts", {})
    for ns, tag in (("dei", "EntityCommonStockSharesOutstanding"), ("us-gaap", "CommonStockSharesOutstanding")):
        rows = f.get(ns, {}).get(tag, {}).get("units", {}).get("shares", [])
        if rows:
            s = pd.DataFrame(rows)[["end", "val", "filed", "form"]]
            s["source"] = f"{ns}:{tag}"
            return s
    return pd.DataFrame()

shares_frames, no_shares = [], []
for i, row in enumerate(universe.itertuples(), 1):
    s = shares_history(row.cik)
    if len(s):
        s.insert(0, "ticker", row.ticker)
        shares_frames.append(s)
    else:
        no_shares.append(row.ticker)
    if i % 25 == 0:
        print(f"{i}/{len(universe)} companies")

shares = pd.concat(shares_frames, ignore_index=True)
shares["filed"] = pd.to_datetime(shares["filed"])
shares["end"] = pd.to_datetime(shares["end"])
# one value per filing date: if several, take the latest period end
shares = (shares.sort_values(["ticker", "filed", "end"])
                .groupby(["ticker", "filed"], as_index=False).last())
shares.to_parquet(OUT / "shares_outstanding.parquet", index=False)
print(f"Shares outstanding for {shares.ticker.nunique()} tickers | none found: {no_shares}")

## 7. Market cap + size bucket

For each trading day, use the **latest share count already filed** on or before that day, times the raw close.

Some companies have **dual-class shares** (e.g. H, EVR, WSO) - SEC's free API leaves their per-class counts out.
For tickers still missing most of their market cap, it falls back to **Yahoo's historical share counts**
(`shares_source = "yahoo"`; slightly less strict on timing, fine for size buckets).

Size buckets are **thirds of the universe by market cap on each date** (`SIZE_METHOD = "tercile"`), so they don't drift as the whole market grows.

In [ ]:
px = px.sort_values(["date", "ticker"])
sh = shares[["ticker", "filed", "end", "val"]].rename(columns={"filed": "date", "end": "shares_as_of"}).sort_values(["date", "ticker"])
px = pd.merge_asof(px, sh, on="date", by="ticker", direction="backward")

# A share count is in "as of its period end" units. If a split happened between then and today,
# scale it so shares and raw_close are on the same basis.
sf = px[["ticker", "date", "split_factor"]].rename(columns={"date": "shares_as_of", "split_factor": "sf_as_of"})
tmp = px[["ticker", "shares_as_of"]].reset_index().dropna(subset=["shares_as_of"]).sort_values("shares_as_of")
tmp = pd.merge_asof(tmp, sf.sort_values("shares_as_of"), on="shares_as_of", by="ticker", direction="nearest")
px["sf_as_of"] = tmp.set_index("index")["sf_as_of"]
px["shares_outstanding"] = px["val"] * px["sf_as_of"] / px["split_factor"]
px["shares_source"] = np.where(px["shares_outstanding"].notna(), "sec", None)
px = px.drop(columns=["val", "sf_as_of"])

# --- Yahoo fallback for tickers still mostly missing (dual-class, FDIC filers, ...)
def yahoo_shares(t):
    try:
        s_ = yf.Ticker(t).get_shares_full(start=PRICE_START)
    except Exception:
        return pd.Series(dtype=float)
    if s_ is None or len(s_) == 0:
        return pd.Series(dtype=float)
    s_ = s_[~s_.index.duplicated(keep="last")].sort_index()
    s_.index = pd.to_datetime(s_.index).tz_localize(None).normalize()
    s_ = s_[~s_.index.duplicated(keep="last")]
    med = s_.rolling(9, center=True, min_periods=1).median()
    return s_[(s_ / med).between(0.5, 2.0)]            # drop Yahoo's occasional spikes

missing_share = px.groupby("ticker")["shares_outstanding"].apply(lambda x: x.isna().mean())
fallback = missing_share[missing_share > 0.2].index.tolist()
print(f"Yahoo share-count fallback for {len(fallback)} tickers: {fallback}")
for t in fallback:
    ys = yahoo_shares(t)
    time.sleep(0.5)
    if ys.empty:
        continue
    m = px["ticker"] == t
    sub = px.loc[m, ["date"]].reset_index()
    ysf = ys.rename("y").rename_axis("date").reset_index()
    sub = pd.merge_asof(sub.sort_values("date"), ysf, on="date", direction="backward")
    sub = sub.set_index("index")["y"]
    fill = m & px["shares_outstanding"].isna() & sub.reindex(px.index).notna()
    px.loc[fill, "shares_outstanding"] = sub.reindex(px.index)[fill]
    px.loc[fill, "shares_source"] = "yahoo"

px["mcap_usd"] = px["raw_close"] * px["shares_outstanding"]

# --- size buckets
if SIZE_METHOD == "tercile":
    px["mcap_bucket"] = px.groupby("date")["mcap_usd"].transform(
        lambda m: pd.qcut(m.rank(method="first"), 3, labels=["small", "mid", "large"]) if m.notna().sum() >= 3 else None)
    px["mcap_bucket"] = px["mcap_bucket"].astype(object)
else:
    def bucket(m):
        if pd.isna(m):
            return None
        for name, (lo, hi) in MCAP_BUCKETS.items():
            if lo <= m < hi:
                return name
    px["mcap_bucket"] = px["mcap_usd"].map(bucket)
px = px.sort_values(["ticker", "date"]).reset_index(drop=True)

latest = px.groupby("ticker").tail(1).merge(universe[["ticker", "bucket"]], on="ticker")
print("Latest market cap (USD bn) by index bucket:")
print((latest.groupby("bucket")["mcap_usd"].describe()[["count", "25%", "50%", "75%"]] / [1, 1e9, 1e9, 1e9]).round(2))
print("\nShare-count source (rows):", px["shares_source"].value_counts(dropna=False).to_dict())
print("Rows missing market cap after fallback:", f"{px.mcap_usd.isna().mean():.1%}")

## 7b. Flag bad price days (`is_bad_price`)

Backtest should **skip** these days. Rules (set in config):
- first `NEW_LISTING_DAYS` trading days of a ticker that starts after the price window opens (IPOs, spin-offs' "when-issued" trading)
- any daily return above `BAD_RETURN` (100%) - at these company sizes almost always a data error
- known broken histories in `BAD_RANGES` (e.g. CHRD before its post-bankruptcy relisting)

In [ ]:
px["bad_price_reason"] = None
first_day = px.groupby("ticker")["date"].transform("min")
day_no = px.groupby("ticker").cumcount()
newly_listed = first_day > pd.Timestamp(PRICE_START) + pd.Timedelta(days=10)
px.loc[newly_listed & (day_no < NEW_LISTING_DAYS), "bad_price_reason"] = "new_listing"
px.loc[px["ret"].abs() > BAD_RETURN, "bad_price_reason"] = "extreme_return"
for t, (a, b) in BAD_RANGES.items():
    m = px["ticker"] == t
    if a: m &= px["date"] >= pd.Timestamp(a)
    if b: m &= px["date"] <= pd.Timestamp(b)
    px.loc[m, "bad_price_reason"] = "known_broken_history"
px["is_bad_price"] = px["bad_price_reason"].notna()

print("Bad price days:", int(px.is_bad_price.sum()), "rows")
print(px.loc[px.is_bad_price].groupby(["bad_price_reason", "ticker"]).size().to_string())

## 8. Data-quality checks

In [ ]:
cov = px.groupby("ticker").agg(first_date=("date", "min"), last_date=("date", "max"), n_days=("date", "count"),
                               mcap_missing=("mcap_usd", lambda s: s.isna().mean()),
                               zero_volume_days=("volume", lambda s: int((s == 0).sum())),
                               extreme_returns=("ret", lambda s: int((s.abs() > EXTREME_RETURN).sum())))
cov["n_calendar_days"] = [((calendar >= a) & (calendar <= b)).sum() for a, b in zip(cov.first_date, cov.last_date)]
cov["missing_days"] = cov["n_calendar_days"] - cov["n_days"]
cov["starts_late"] = cov["first_date"] > pd.Timestamp("2016-01-01")
cov["ends_early"] = cov["last_date"] < calendar[-1] - pd.Timedelta(days=10)
cov = cov.reset_index()
cov.to_csv(OUT / "price_coverage.csv", index=False)

print("Tickers with prices:", len(cov), "of", len(TICKERS), "| missing entirely:", sorted(set(TICKERS) - set(cov.ticker)))
print("Start after 2016 (listed later):", int(cov.starts_late.sum()))
print("Stop early (delisted / acquired):", int(cov.ends_early.sum()))
print("Tickers with missing days:", int((cov.missing_days > 0).sum()))
print("Tickers with extreme daily returns:", int((cov.extreme_returns > 0).sum()))
print("Rows missing market cap:", f"{px.mcap_usd.isna().mean():.1%}")
display(cov.sort_values(["extreme_returns", "missing_days"], ascending=False).head(10)) if "display" in globals() else None

## 8b. Universe quality: which companies to keep, and from when

Combines price coverage, market-cap coverage and **8-K coverage** (from notebook 01's filing index):
- **too few 8-Ks** -> usually a bank that files with the FDIC instead of SEC -> `drop`
- **8-Ks start much later than prices** -> company re-organised under a new SEC ID; earlier filings sit under the old ID -> `keep_from` that date
- otherwise `keep`

Saved as `universe_quality.csv`. **Everyone should filter events and prices with its `usable_from` / `recommendation` columns.**

In [ ]:
q = cov[["ticker", "first_date", "last_date", "mcap_missing"]].rename(columns={"first_date": "price_start"})
q["mcap_missing"] = px.groupby("ticker")["mcap_usd"].apply(lambda s: s.isna().mean()).reindex(q.ticker).values.round(2)
fi_path = BASE / "index" / "filing_index.parquet"
if fi_path.exists():
    fi = pd.read_parquet(fi_path, columns=["ticker", "filing_date", "is_div_candidate", "is_buyback_candidate"])
    fi["filing_date"] = pd.to_datetime(fi["filing_date"])
    per = fi.groupby("ticker").agg(filings=("filing_date", "count"), first_filing=("filing_date", "min"),
                                   last_filing=("filing_date", "max"),
                                   div_candidates=("is_div_candidate", "sum"), buyback_candidates=("is_buyback_candidate", "sum"))
    q = q.merge(per, left_on="ticker", right_index=True, how="left")
else:
    print("filing_index.parquet not found - run notebook 01 first; skipping 8-K coverage")
    q["filings"] = np.nan; q["first_filing"] = pd.NaT

q["filings"] = q["filings"].fillna(0).astype(int)
filings_window_start = max(pd.Timestamp("2016-01-01"), q["price_start"].min())

def assess(r):
    flags = []
    if r.filings < MIN_FILINGS:
        flags.append("too_few_filings")
    elif pd.notna(r.first_filing) and (r.first_filing - max(r.price_start, filings_window_start)).days > LATE_FILINGS_GAP_DAYS:
        flags.append("filings_start_late")
    if r.price_start > filings_window_start:
        flags.append("listed_after_2016")
    if r.mcap_missing > 0.5:
        flags.append("mcap_mostly_missing")
    if r.ticker in BAD_RANGES:
        flags.append("known_broken_history")
    return ";".join(flags)

q["flags"] = q.apply(assess, axis=1)
q["recommendation"] = np.where(q["flags"].str.contains("too_few_filings"), "drop",
                       np.where(q["flags"].str.contains("filings_start_late|known_broken_history|listed_after_2016"), "keep_from", "keep"))

# usable_from = latest of: first filing, price start + new-listing buffer, end of known broken range
buffer = pd.Timedelta(days=int(NEW_LISTING_DAYS * 1.5))
def usable_from(r):
    if r.recommendation == "drop":
        return pd.NaT
    d = max(filings_window_start, r.price_start + (buffer if r.price_start > filings_window_start else pd.Timedelta(0)))
    if "filings_start_late" in r["flags"] and pd.notna(r.first_filing):
        d = max(d, r.first_filing)
    if r.ticker in BAD_RANGES and BAD_RANGES[r.ticker][1]:
        d = max(d, pd.Timestamp(BAD_RANGES[r.ticker][1]) + pd.Timedelta(days=1))
    return d
q["usable_from"] = q.apply(usable_from, axis=1)

q.to_csv(BASE / "universe_quality.csv", index=False)
print(q["recommendation"].value_counts().to_string())
print("\nNot a plain keep:")
show = q[q.recommendation != "keep"].sort_values(["recommendation", "ticker"])
print(show[["ticker", "recommendation", "usable_from", "filings", "first_filing", "price_start", "mcap_missing", "flags"]].to_string(index=False))

## 9. Save

In [ ]:
keep = ["date", "ticker", "open", "high", "low", "close", "adj_close", "volume", "ret",
        "raw_close", "split_factor", "shares_outstanding", "shares_source", "mcap_usd", "mcap_bucket",
        "is_bad_price", "bad_price_reason"]
px[keep].to_parquet(OUT / "prices.parquet", index=False)
bm[["date", "ticker", "open", "high", "low", "close", "adj_close", "volume", "ret"]].to_parquet(OUT / "benchmarks.parquet", index=False)
px.loc[px["dividend"].fillna(0) > 0, ["date", "ticker", "dividend"]].rename(columns={"date": "ex_date"}) \
  .to_parquet(OUT / "dividends_yf.parquet", index=False)
splits.to_parquet(OUT / "splits_yf.parquet", index=False)
for f in sorted(OUT.glob("*")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.1f} MB")

## 10. Helpers for the team (copy into `src/store/`)

`load_prices()` / `load_benchmarks()` / `first_tradeable_date()` — everyone should use these instead of reading files directly.
`first_tradeable_date` takes the SEC acceptance time **in US Eastern** and returns the first day a trade is allowed.

In [ ]:
from datetime import time as dtime

def load_prices(tickers=None, start=None, end=None, drop_bad=True, usable_only=True):
    """drop_bad: remove is_bad_price days. usable_only: apply universe_quality.csv (drop + usable_from)."""
    df = pd.read_parquet(OUT / "prices.parquet")
    if drop_bad:
        df = df[~df["is_bad_price"]]
    if usable_only and (BASE / "universe_quality.csv").exists():
        uq = pd.read_csv(BASE / "universe_quality.csv", parse_dates=["usable_from"])
        uq = uq[uq.recommendation != "drop"][["ticker", "usable_from"]]
        df = df.merge(uq, on="ticker")
        df = df[df.date >= df.usable_from].drop(columns="usable_from")
    if tickers is not None: df = df[df.ticker.isin(tickers)]
    if start is not None:   df = df[df.date >= pd.Timestamp(start)]
    if end is not None:     df = df[df.date <= pd.Timestamp(end)]
    return df

def load_benchmarks(ticker="SPY"):
    df = pd.read_parquet(OUT / "benchmarks.parquet")
    return df[df.ticker == ticker]

CAL = pd.DatetimeIndex(pd.read_csv(OUT / "trading_calendar.csv", parse_dates=["date"])["date"])
MARKET_CLOSE = dtime(16, 0)

def first_tradeable_date(acceptance_et):
    """Same trading day if accepted before 16:00 ET on a trading day, else the next trading day."""
    ts = pd.Timestamp(acceptance_et)
    day = ts.normalize().tz_localize(None) if ts.tzinfo else ts.normalize()
    if day in CAL and ts.time() < MARKET_CLOSE:
        return day.date()
    nxt = CAL[CAL > day]
    return nxt[0].date() if len(nxt) else None

# quick checks
print(first_tradeable_date("2024-01-22 15:59"))   # Mon before close -> same day
print(first_tradeable_date("2024-01-22 16:05"))   # after close      -> Tue
print(first_tradeable_date("2024-01-20 10:00"))   # Saturday         -> Mon

## 11. Make sure everything reaches Google Drive (run before closing the tab)

In [ ]:
if MOUNTED:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files synced to Drive.")

## Notes for the report
- **Survivorship bias:** the universe is *current* S&P 400/600 members, so companies delisted since 2016 are missing.
- **Market cap** uses share counts from 10-K/10-Q cover pages, dated by filing date (point-in-time); dual-class companies fall back to Yahoo share counts (`shares_source`).
- **Size buckets** are terciles of the universe by market cap on each date.
- **Excluded data:** `is_bad_price` days (new listings, >100% daily moves, CHRD's bankruptcy period) and companies marked `drop` in `universe_quality.csv` (too few 8-Ks, e.g. FDIC-filing banks). Companies that changed SEC ID are used only from `usable_from`.
- **Yahoo** data is free and not survivorship-free; check `price_coverage.csv` for flagged tickers before using them.